# Trafic fines package

Instrucciones y ejemplos de uso para el paquete de `traficFines`.

## Cache

Consideracines de diseño para la clase `Cache`:

- Todos los atributos: `app_name`, `obsolescence` y `cache_dir` son privados, se pueden acceder a ellos a través de métodos públicos declarados con `@property` y no se pueden modificar después de la inicialización ya que no se han declarado métodos `@setter` para estos atributos, intentar assignar un valor lanza una excepción `AttributeError` indicando que el atributo no se puede modificar.
- Los métodos `exists`, `load`, `how_old` y `delete` tienen que ser públicos según los requerimientos del ejercicio, ya que se espera que estos métodos sean redefinidos en la clase `CacheURL`.
- El resto de métodos `set` y `clear` también se han declarado como públicos.
- Se define un método privado `_get_file_age` que devuelve la edad del fichero como un objeto de tipo `datetime.timedelta`.
    * El método trata de consultar la fecha de creación del fichero utilizando `stat().st_birthtime`, pero si esta información no está disponible (lo que puede ocurrir en algunos sistemas operativos), utiliza la fecha de última modificación con `stat().st_mtime` como alternativa.
- El método `load`:
    * Lanza una excepción `CacheError` si el fichero en caché no existe, también comprueba si el fichero está obsoleto y en ese caso también lanza una excepción `CacheError` indicando que el fichero está obsoleto.
    * Utiliza el método privado `_get_file_age` para comprobar la obsolescencia del fichero, no se utiliza el método `how_old` ya que este puede ser redefinido en las clases hijas y dar algunos problemas.
- El método `clear` borra todos los ficheros de caché que existen en la aplicación `app_name`.

**NOTA:**

- `st_birthtime`: Se añade en `ptyhon 3.12` no siempre disponible.\
                  **Ver:** https://docs.python.org/3.12/library/os.html#os.stat_result.st_birthtime
- `st_mtime`: Fecha de última modificación del fichero, se utiliza como alternativa a `st_birthtime` si esta no está disponible.

- No se utiliza `st_ctime` como alternativa a `st_birthtime` ya que desde `python 3.12` está deprecado en Windows.\
  **Ver:** https://docs.python.org/3.12/library/os.html#os.stat_result.st_ctime

**POSIBLES MEJORAS:**

- El método `set` recibe el nombre del fichero que se va almacenar en caché y almacena los datos que recibe como un `str`. Si los datos son muy grandes esto puede suponer un problema de memoria.
  
  Como mejora, se puede crear un método alternativo `set_from_file` que guarde los datos de `file-like object`, que permita escribir los datos en cache de forma incremental y mejorar la eficiencia en memoria.

In [2]:
from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd().parent / "src" / "traficFines"))
sys.path


['/home/cora/Documents/data-engineer/moduloI/Tarea/tarea/src/traficFines',
 '/home/cora/.local/share/uv/python/cpython-3.12.11-linux-x86_64-gnu/lib/python312.zip',
 '/home/cora/.local/share/uv/python/cpython-3.12.11-linux-x86_64-gnu/lib/python3.12',
 '/home/cora/.local/share/uv/python/cpython-3.12.11-linux-x86_64-gnu/lib/python3.12/lib-dynload',
 '',
 '/home/cora/Documents/data-engineer/moduloI/Tarea/tarea/.venv/lib/python3.12/site-packages']

Para el ejemplo de uso instanciamos la clase `Cache` con un nombre de aplicación **"test_cache"** y un tiempo de obsolescencia de **1 día**.

In [3]:
from cache import Cache

cache = Cache(app_name="test_cache", obsolescence=1)
cache


Cache(app_name='test_cache', obsolescence=1)

In [4]:
# Comprobamos que el directorio de la aplicación "test_cache" se ha creado correctamente
!ls -lhF ~/.my_cache


total 0
drwxr-xr-x 1 cora cora 0 Apr  1 14:15 test_cache/


Los atributos de la clase `Cache` son privados, no se ha definido se ha definido `@property` para acceder a ellos, por lo que no se pueden consultar directamente. Sin embargo, no se han definido los setters para que no puedan modificarse directamente.

In [5]:
print("Atributos de la caché:")
print(f"  - app_name: {cache.app_name}")
print(f"  - obsolescence: {cache.obsolescence} días")
print(f"  - cache_dir: {cache.cache_dir}")


Atributos de la caché:
  - app_name: test_cache
  - obsolescence: 1 días
  - cache_dir: /home/cora/.my_cache/test_cache


In [6]:
# Si intentamos setear un valor, se lanzará una excepción de tipo AttributeError porque el método set no está implementado
cache.app_name = "otra_cache"


AttributeError: property 'app_name' of 'Cache' object has no setter

In [7]:
# Ahora mismo no hay nada en la caché
print(cache)


La caché de 'test_cache' contiene 0 archivos.


In [8]:
# Cargamos un par de fichero en la caché
cache.set("trabaluengas", "Tres tristes tigres tragan trigo en un trigal")
cache.set("refran", "Quién fue a Sevilla perdió su silla")

print(cache)


La caché de 'test_cache' contiene 2 archivos.


In [9]:
!ls -lhF ~/.my_cache/test_cache


total 8.0K
-rw-r--r-- 1 cora cora 37 Apr  1 14:15 refran
-rw-r--r-- 1 cora cora 45 Apr  1 14:15 trabaluengas


Lectura de un fichero de la caché:

In [10]:
cache.load("trabaluengas")


0:00:05.844193 1 day, 0:00:00


'Tres tristes tigres tragan trigo en un trigal'

Si el fichero no existe, se lanza una excepción de tipo `CacheError`:

In [11]:
cache.load("no_existe")


CacheError: El archivo de caché 'no_existe' no existe.

¿Qué pasa si el fichero existe pero la obsolescencia ha pasado?

**NOTA:**

Ahora la obsolescencia es de 1 día, vamos a aplicar hackear los ficheros para que parezca que se crearon antes y luego intentamos cargarlos.

In [12]:
!stat  ~/.my_cache/test_cache/refran


  File: /home/cora/.my_cache/test_cache/refran
  Size: 37        	Blocks: 8          IO Block: 4096   regular file
Device: 0,55	Inode: 14364863    Links: 1
Access: (0644/-rw-r--r--)  Uid: ( 1000/    cora)   Gid: ( 1000/    cora)
Access: 2026-04-01 14:15:17.839260724 +0200
Modify: 2026-04-01 14:15:17.839422198 +0200
Change: 2026-04-01 14:15:17.839422198 +0200
 Birth: 2026-04-01 14:15:17.839260724 +0200


In [13]:
# Simulamos que el fichero "refran" se ha quedado obsoleto
!touch -d "2026-02-01 02:14:31.196623114" ~/.my_cache/test_cache/refran
!stat  ~/.my_cache/test_cache/refran


  File: /home/cora/.my_cache/test_cache/refran
  Size: 37        	Blocks: 8          IO Block: 4096   regular file
Device: 0,55	Inode: 14364863    Links: 1
Access: (0644/-rw-r--r--)  Uid: ( 1000/    cora)   Gid: ( 1000/    cora)
Access: 2026-02-01 02:14:31.196623114 +0100
Modify: 2026-02-01 02:14:31.196623114 +0100
Change: 2026-04-01 14:16:10.897886665 +0200
 Birth: 2026-04-01 14:15:17.839260724 +0200


In [14]:
cache.load("refran")


59 days, 11:01:43.275668 1 day, 0:00:00


CacheError: El archivo de caché 'refran' está obsoleto (edad: 59 days, 11:01:43.275668).

Comprobamos el resto de métodos `exists`, `how_old`, `delete`y `clear`.

In [15]:
cache.exists("trabaluengas")


True

In [ ]:
import datetime as dt

age = cache.how_old("refran")
print(f"El fichero 'refran' tiene {age} milisegundos de edad.")
# Mostramos la edad del fichero "refran" en un formato más legible utilizando timedelta
print(f"El fichero 'refran' tiene {dt.timedelta(milliseconds=age)} de edad.")


El fichero 'refran' tiene 5137435652.897 milisegundos de edad.
El fichero 'refran' tiene 59 days, 11:03:55.652897 de edad.


In [18]:
cache.delete("refran")


In [19]:
!ls -lhF ~/.my_cache/test_cache


total 4.0K
-rw-r--r-- 1 cora cora 45 Apr  1 14:15 trabaluengas


In [ ]:
print(cache)


La caché de 'test_cache' contiene 1 archivo.


In [21]:
# El método clear elimina todos los ficheros de la caché de la aplicación
cache.clear()


In [22]:
!ls -lhF ~/.my_cache/test_cache


total 0


## CacheURL

Consideracines de diseño para la clase `CacheURL`:

- Se define la clase `@staticmethod` privada `_url_to_filename` para convertir una URL en un nombre de fichero seguro para la caché utilizando hashing **md5**.
- Los métodos `exists`, `load`, `delete` y `how_old` de la clase `CacheURL` se redefinen para trabajar con URL's:
    * La reimplementación consiste en convertir la URL a un nombre de fichero válido utilizando `_url_to_filename` y luego delegar la operación a los métodos correspondientes de la clase padre `Cache`.
- El método `get` trata de recuperar el contenido de la URL desde la caché, si existe y no está obsoleto, si no, realiza una petición para obtener los datos actualizados y los guarda en la caché.
    * Sólo convierte la URL a nombre de fichero una vez al principio del método, evitando conversiones repetidas (se utiliza el nombre del hasheado en los métodos de la clase padre).
    * Si la petición a la URL falla, `raise_for_status()` lanza una exepción `HTTPError`.



Para el ejemplo de uso instanciamos la clase `CacheURL` con un nombre de aplicación **"test_cache_url"** y un tiempo de obsolescencia de **1 día**.

In [29]:
from cache import CacheURL

cache = CacheURL(app_name="test_cache_url", obsolescence=1)
cache


Cache(app_name='test_cache_url', obsolescence=1)

In [30]:
# Comprobamos que el directorio de la aplicación "test_cache_url" se ha creado correctamente
!ls -lhF ~/.my_cache


total 0
drwxr-xr-x 1 cora cora 0 Apr  1 14:19 test_cache/
drwxr-xr-x 1 cora cora 0 Apr  1 17:17 test_cache_url/


In [ ]:
print(cache)


La caché de 'test_cache_url' contiene 0 archivos.


Traemos el contenido de las webs: http://example.com y [hacker news](https://news.ycombinator.com/), la primera vez no están en caché así que las descargará de la red.

In [ ]:
print(cache.get("http://example.com"))


<!doctype html><html lang="en"><head><title>Example Domain</title><meta name="viewport" content="width=device-width, initial-scale=1"><style>body{background:#eee;width:60vw;margin:15vh auto;font-family:system-ui,sans-serif}h1{font-size:1.5em}div{opacity:0.8}a:link,a:visited{color:#348}</style></head><body><div><h1>Example Domain</h1><p>This domain is for use in documentation examples without needing permission. Avoid use in operations.</p><p><a href="https://iana.org/domains/example">Learn more</a></p></div></body></html>



In [ ]:
print(cache.get("https://news.ycombinator.com/"))


<html lang="en" op="news"><head><meta name="referrer" content="origin"><meta name="viewport" content="width=device-width, initial-scale=1.0"><link rel="stylesheet" type="text/css" href="news.css?mqRJVV13UcCuW3P4GUfP"><link rel="icon" href="y18.svg"><link rel="alternate" type="application/rss+xml" title="RSS" href="rss"><title>Hacker News</title></head><body><center><table id="hnmain" border="0" cellpadding="0" cellspacing="0" width="85%" bgcolor="#f6f6ef"><tr><td bgcolor="#ff6600"><table border="0" cellpadding="0" cellspacing="0" width="100%" style="padding:2px"><tr><td style="width:18px;padding-right:4px"><a href="https://news.ycombinator.com"><img src="y18.svg" width="18" height="18" style="border:1px white solid; display:block"></a></td><td style="line-height:12pt; height:10px;"><span class="pagetop"><b class="hnname"><a href="news">Hacker News</a></b><a href="newest">new</a> | <a href="front">past</a> | <a href="newcomments">comments</a> | <a href="ask">ask</a> | <a href="show">sho

In [ ]:
print(cache)


La caché de 'test_cache_url' contiene 2 archivos.


In [ ]:
!ls -lhF ~/.my_cache/test_cache_url


total 40K
-rw-r--r-- 1 cora cora 528 Apr  1 17:17 a9b9f04336ce0181a08e774e01113b31
-rw-r--r-- 1 cora cora 34K Apr  1 17:18 cbc580718eb38c03483c4d7616fb220d


In [ ]:
# Comprobamos que el nombre de la URLs está bien hasheado
!echo -n "http://example.com" | md5sum
!echo -n "https://news.ycombinator.com/" | md5sum


a9b9f04336ce0181a08e774e01113b31  -
cbc580718eb38c03483c4d7616fb220d  -


Ahora los ficheros están en caché, si hacemos un `get` de nuevo, no se volverán a cargar, si no que se cargarán desde la caché.

**NOTA**:
Para validarlo, vamos a comprobar la fecha de modificación de los ficheros, antes y después de hacer un `get`.

In [ ]:
!stat "$HOME/.my_cache/test_cache_url/$(echo -n 'http://example.com' | md5sum | awk -F ' ' '{print $1}')"


  File: /home/cora/.my_cache/test_cache_url/a9b9f04336ce0181a08e774e01113b31
  Size: 528       	Blocks: 8          IO Block: 4096   regular file
Device: 0,55	Inode: 14368205    Links: 1
Access: (0644/-rw-r--r--)  Uid: ( 1000/    cora)   Gid: ( 1000/    cora)
Access: 2026-04-01 17:17:54.764671576 +0200
Modify: 2026-04-01 17:17:54.765568623 +0200
Change: 2026-04-01 17:17:54.765568623 +0200
 Birth: 2026-04-01 17:17:54.764671576 +0200


In [ ]:
print(cache.get("http://example.com"))


0:11:23.683666 1 day, 0:00:00
<!doctype html><html lang="en"><head><title>Example Domain</title><meta name="viewport" content="width=device-width, initial-scale=1"><style>body{background:#eee;width:60vw;margin:15vh auto;font-family:system-ui,sans-serif}h1{font-size:1.5em}div{opacity:0.8}a:link,a:visited{color:#348}</style></head><body><div><h1>Example Domain</h1><p>This domain is for use in documentation examples without needing permission. Avoid use in operations.</p><p><a href="https://iana.org/domains/example">Learn more</a></p></div></body></html>



In [ ]:
!stat "$HOME/.my_cache/test_cache_url/$(echo -n 'http://example.com' | md5sum | awk -F ' ' '{print $1}')"


  File: /home/cora/.my_cache/test_cache_url/a9b9f04336ce0181a08e774e01113b31
  Size: 528       	Blocks: 8          IO Block: 4096   regular file
Device: 0,55	Inode: 14368205    Links: 1
Access: (0644/-rw-r--r--)  Uid: ( 1000/    cora)   Gid: ( 1000/    cora)
Access: 2026-04-01 17:29:18.448641710 +0200
Modify: 2026-04-01 17:17:54.765568623 +0200
Change: 2026-04-01 17:17:54.765568623 +0200
 Birth: 2026-04-01 17:17:54.764671576 +0200


Para simular que el fichero está obsoleto, podemos modificar la fecha de modificación de la URL "http://example.com" de forma manual. Al hacer el `get` tendrá que descargar datos nuevos y guardarlos en caché.

In [ ]:
!touch -d "2026-02-01 02:14:31.196623114" "$HOME/.my_cache/test_cache_url/$(echo -n 'http://example.com' | md5sum | awk -F ' ' '{print $1}')"
!stat "$HOME/.my_cache/test_cache_url/$(echo -n 'http://example.com' | md5sum | awk -F ' ' '{print $1}')"


  File: /home/cora/.my_cache/test_cache_url/a9b9f04336ce0181a08e774e01113b31
  Size: 528       	Blocks: 8          IO Block: 4096   regular file
Device: 0,55	Inode: 14368205    Links: 1
Access: (0644/-rw-r--r--)  Uid: ( 1000/    cora)   Gid: ( 1000/    cora)
Access: 2026-02-01 02:14:31.196623114 +0100
Modify: 2026-02-01 02:14:31.196623114 +0100
Change: 2026-04-01 17:34:43.024346709 +0200
 Birth: 2026-04-01 17:17:54.764671576 +0200


In [ ]:
from time import sleep

# Ahora el fichero de la URL "http://example.com" se ha quedado obsoleto, tiene que descargar datos nuevos
print(cache.get("http://example.com"))

# Hacemos un sleep de 10 segundos
sleep(10)

# Consultamos la edad del fichero que tendrá que ser de aproximadamente 10 segundos (10_000 milisegundos)
age = cache.how_old("http://example.com")
print(f"El fichero de la URL 'http://example.com' tiene {age} milisegundos de edad.")
print(
    f"El fichero de la URL 'http://example.com' tiene {dt.timedelta(milliseconds=age)} de edad."
)


59 days, 14:22:09.625780 1 day, 0:00:00
<!doctype html><html lang="en"><head><title>Example Domain</title><meta name="viewport" content="width=device-width, initial-scale=1"><style>body{background:#eee;width:60vw;margin:15vh auto;font-family:system-ui,sans-serif}h1{font-size:1.5em}div{opacity:0.8}a:link,a:visited{color:#348}</style></head><body><div><h1>Example Domain</h1><p>This domain is for use in documentation examples without needing permission. Avoid use in operations.</p><p><a href="https://iana.org/domains/example">Learn more</a></p></div></body></html>

El fichero de la URL 'http://example.com' tiene 10001.6 milisegundos de edad.
El fichero de la URL 'http://example.com' tiene 0:00:10.001600 de edad.


Sólo queda por probar el resto de métodos que se han sobreescrito.

In [ ]:
cache.exists("https://news.ycombinator.com/")


True

In [ ]:
print(cache.load("https://news.ycombinator.com/"))


0:20:29.172022 1 day, 0:00:00
<html lang="en" op="news"><head><meta name="referrer" content="origin"><meta name="viewport" content="width=device-width, initial-scale=1.0"><link rel="stylesheet" type="text/css" href="news.css?mqRJVV13UcCuW3P4GUfP"><link rel="icon" href="y18.svg"><link rel="alternate" type="application/rss+xml" title="RSS" href="rss"><title>Hacker News</title></head><body><center><table id="hnmain" border="0" cellpadding="0" cellspacing="0" width="85%" bgcolor="#f6f6ef"><tr><td bgcolor="#ff6600"><table border="0" cellpadding="0" cellspacing="0" width="100%" style="padding:2px"><tr><td style="width:18px;padding-right:4px"><a href="https://news.ycombinator.com"><img src="y18.svg" width="18" height="18" style="border:1px white solid; display:block"></a></td><td style="line-height:12pt; height:10px;"><span class="pagetop"><b class="hnname"><a href="news">Hacker News</a></b><a href="newest">new</a> | <a href="front">past</a> | <a href="newcomments">comments</a> | <a href="ask

In [ ]:
cache.delete("https://news.ycombinator.com/")


In [ ]:
print(cache)


La caché de 'test_cache_url' contiene 1 archivo.


In [ ]:
# Ahora que hemos borrado el fichero, load tendrá que lanzar una excepción de tipo CacheError porque el fichero no existe
cache.load("https://news.ycombinator.com/")


CacheError: El archivo de caché 'cbc580718eb38c03483c4d7616fb220d' no existe.

## MadridFines